
## Problema 5
Aplica el método de la potencia y potencia inversa para el cálculo del autovalor y autovector dominante de la matriz propuesta y su inversa
$$A=\begin{pmatrix}15&-2&2\\
1&10&-3\\
-2&1&0 \end{pmatrix}$$

En el siguiente algoritmo del método de la potencia (con normalización, nuestro programa recibirá las siguientes entradas
- $A$ := Matriz de interés
- $x$ := $(1,0,0)$, Vector inicial
- $t$ := $0.001$, Tolerancia
- $M$ := 100, Número máximo de iteraciones
Definiremos a la forma lineal $\varphi(x)$ como $$\varphi(x)=x_1$$

In [33]:
# Declaramos la entrada inicial como

A = Float32[15  -2   2;
            1   10  -3;
            -2  1   0]

x0 = Float64[1, 0, 0] # Vector Inicial
t = 0.001 # Usaremos una tolorencia de 10^-2
M = 1000 # Máximo de iteraciones

# Evaluamos la función
# Los resultado de la tabla estan redondeados a 4 cifras significativas
PowerMethod(A, x0, t, M)

          Método de la potencia con  normalización
 ------------ ------------ ------------------------ --------
  #Iteración   Eigenvalor              Eigenvector    Error 
 ------------ ------------ ------------------------ --------
           0          0.0          [1.0, 0.0, 0.0]        1
           1         15.0   [1.0, 0.0667, -0.1333]   1.0667
           2         14.6   [1.0, 0.1416, -0.1324]   0.7461
          10      14.1404   [1.0, 0.3153, -0.1195]   0.0532
          11      14.1304   [1.0, 0.3193, -0.1192]    0.039
          12       14.123    [1.0, 0.3222, -0.119]   0.0286
          19      14.1049   [1.0, 0.3294, -0.1185]   0.0033
          20      14.1043   [1.0, 0.3296, -0.1184]   0.0025
          22      14.1035   [1.0, 0.3299, -0.1184]   0.0013
          23      14.1033     [1.0, 0.33, -0.1184]    0.001
 ------------ ------------ ------------------------ --------
Tolerancia =.001 Vector, inicial = (1,0,0)


Calcularemos ahora el vector el eigenvalor máximo de $A^{-1}$, que corresponde al inverso del eigenvalor de $A$.
Para el cálculo del vector $x^{(k+1)}$ en el método de la potencia aplicado a $A^{-1}$ tenemos que $$x^{(k+1)}=A^{-1}x^{(k)}$$ es decir $Ax^{(k+1)}=LUx^{(k+1)}=x^{(k)}$ de donde podemos resolver el sistema 
$$Ux^{(k+1)}=L^{-1}x^{(k)}$$
para obtener el valor del siguiente vector en la iteración. Simbólicamente podemos escribir al vector $x^{k+1}$ como la división de $U$ por la izquierda
$$x^{(k+1)}=U\ \backslash\ L^{-1}x^{(k)}$$
Notemos también que el vector error dado por $v=A^{-1}x-rx$, dado que nos gustaría que $A^{-1}x\approx rx$ o equivalentemente que $$x\approx rAx$$
puede sustituirse entonces por el nuevo vector error $$\tilde{v}= x - rAx$$ lo cual nos ahorra, nuevamente, calcular la inversa de A.

 Nuestro programa recibirá las siguientes entradas
- $A$ := Matriz de interés
- $U$ := Matriz triangular superior de la factorización $LU$ de $A$
- $L^{-1}$ := Matriz triangular inferior de la factorización $LU$ de $A$
- $x$ := $(1,0,0)$, Vector inicial
- $t$ := $0.001$, Tolerancia
- $M$ := 100, Número máximo de iteraciones
Definiremos a la forma lineal $\varphi(x)$ como $$\varphi(x)=x_1$$

In [32]:
A = Float32[15  -2   2;
            1   10  -3;
            -2  1   0]

factores = factorize(A)
U = factores.U
L_inv = inv(factores.L)

x0 = Float64[1, 0, 0] # Vector Inicial
t = 0.001 # Usaremos una tolorencia de 10^-2
M = 1000 # Máximo de iteraciones

# Evaluamos la función
# Los resultado de la tabla estan redondeados a 4 cifras significativas
InversePowerMethod(A, U, L_inv, x0, t, M)


                Método de la potencia inversa
 ------------ ------------ ------------------------ --------
  #Iteración   Eigenvalor              Eigenvector    Error 
 ------------ ------------ ------------------------ --------
           0          0.0          [1.0, 0.0, 0.0]        1
           1         0.04    [0.1429, 0.2857, 1.0]      1.0
           2      -1.2133   [-0.0856, 0.3227, 1.0]   1.5992
           3         2.12   [-0.0928, 0.3258, 1.0]   0.0842
           4       1.9576     [-0.093, 0.326, 1.0]   0.0024
           5       1.9529     [-0.093, 0.326, 1.0]   0.0001
 ------------ ------------ ------------------------ --------
Tolerancia =.001 Vector, inicial = (1,0,0)


### Conclusiones
La matriz $A$ tiene como eigenvalor y vectores dominantes a
$$\lambda=14.1033\quad\text{y}\quad v=(1.0, 0.33, -0.1184)$$

Por otro lado, la matriz $A^{-1}$ tiene como eigenvalor y vectores dominantes a
$$\tilde\lambda=1.9529\quad\text{y}\quad \tilde v=(-0.093, 0.326, 1.0)$$

### Número de condición 

Dado que $\rho(A) = \lambda$ y $\rho(A)=\tilde\lambda$ (los radios espectrales) son los ínfimos, tomados entre todas las normas de $A$ y de $A^{-1}$. Entonces podemos aproximar $\kappa(A)$ como
$$\kappa(A)=||A||\ ||A^{-1}||\approx\lambda\tilde\lambda\approx27.5423$$
en donde el valor anterior es tambien una muy buena aproximación al ínfimo de $\kappa(A)$.

### Codigos de los algoritmos utilizados

#### Método de la Potencia con normalización

In [31]:
# Importamos librerias
using LinearAlgebra
using PrettyTables

# Método de la potencia con normalización

phi(x) = x[1] 

function PowerMethod(A, x, t, M)
    r = 0.0
    error = 1

    # Arrelgo para guardar las iteraciones
    Iter = [0 r [x] error]

    for i = 1:M
        y = A * x
        r = phi(y) / phi(x)
        # Normalizamos con la norma infinito
        x = y / norm(y, Inf)     

        v = A * x - r * x # Vector error
        # Calculo del error con la norma infinito
        error = norm(v, Inf)

        # Actualizamos el arreglo 
        # Los datos se redondean a 4 cifras significativas para su visualización
        Iter = vcat(Iter, [i round(r, digits=4) [round.(x, digits=4)] round(error, digits=4)]) 

        # Interrumpimos el ciclo al alcanzar la tolerancia
        if error < t
            break
        end

    end

    # Devolvemos los datos de las iteraciones
    pretty_table(Iter[[1,2,3,11,12,13,20,21,23,24], :]; 
            column_labels = [ "#Iteración", "Eigenvalor", "Eigenvector", "Error"],
            title = "Método de la potencia con  normalización",
            backend = :text,
            table_format = TextTableFormat(borders = text_table_borders__compact),
            source_notes = "Tolerancia =.001 Vector, inicial = (1,0,0)")   
end;

#### Método de la Potencia Inversa

In [30]:
# Método de la potencia para la inversa

function InversePowerMethod(A, U, L_inv, x, t, M)
    # Inicializamos los valores de r y del error
    r = 0.0
    error = 1

    # Arrelgo para guardar las iteraciones
    Iter = [0 r [x] error]

    for i = 1:M
        y = U \ (L_inv * x)
        r = phi(y) / phi(x)
        # Normalizamos con la norma infinito
        x = y / norm(y, Inf)     

        v = x - r * A * x # Vector error
        # Calculo del error con la norma infinito
        error = norm(v, Inf)

        # Actualizamos el arreglo 
        # Los datos se redondean a 4 cifras significativas para su visualización
        Iter = vcat(Iter, [i round(r, digits=4) [round.(x, digits=4)] round(error, digits=4)]) 

        # Interrumpimos el ciclo al alcanzar la tolerancia
        if error < t
            break
        end

    end

    # Devolvemos los datos de las iteraciones
    pretty_table(Iter; 
            column_labels = [ "#Iteración", "Eigenvalor", "Eigenvector", "Error"],
            title = "Método de la potencia inversa",
            backend = :text,
            table_format = TextTableFormat(borders = text_table_borders__compact),
            source_notes = "Tolerancia =.001 Vector, inicial = (1,0,0)")    
end;